In [1]:
import pandas as pd
import tiktoken
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset

In [5]:
# 1. Load dataset & subsample
df = pd.read_csv("IMDB Dataset.csv", engine="python", on_bad_lines="skip")
df = df.sample(n=2500, random_state=42).reset_index(drop=True)

In [6]:
# 2. Convert sentiment column to numeric (positive -> 1, negative -> 0)
df["sentiment"] = df["sentiment"].map({"positive": 1, "negative": 0})

In [7]:
# 3. Split data into train and test sets
train_texts, test_texts, train_labels, test_labels = train_test_split(
    df["review"].tolist(),
    df["sentiment"].tolist(),
    test_size=0.2,
    random_state=42,
)

In [8]:
# 4. Tokenize using tiktoken (cl100k_base)
enc = tiktoken.get_encoding("cl100k_base")
train_tokens = [enc.encode(text) for text in train_texts]
test_tokens = [enc.encode(text) for text in test_texts]

In [9]:
# 5. Pad or truncate sequences to a fixed length
MAX_LEN = 200
PAD_TOKEN = 0  # 0 is safe to use for padding


def pad_truncate(seq, max_len=MAX_LEN):
    if len(seq) > max_len:
        return seq[:max_len]
    return seq + [PAD_TOKEN] * (max_len - len(seq))


train_padded = [pad_truncate(seq) for seq in train_tokens]
test_padded = [pad_truncate(seq) for seq in test_tokens]

In [10]:
# 6. Convert sequences and labels into PyTorch tensors & create DataLoaders
X_train = torch.tensor(train_padded, dtype=torch.long)
y_train = torch.tensor(train_labels, dtype=torch.float32)
X_test = torch.tensor(test_padded, dtype=torch.long)
y_test = torch.tensor(test_labels, dtype=torch.float32)

train_loader = DataLoader(
    TensorDataset(X_train, y_train), batch_size=32, shuffle=True
)
test_loader = DataLoader(TensorDataset(X_test, y_test), batch_size=32)

In [11]:
# 7. Build RNN Model: Embedding -> nn.RNN -> Linear output layer
class RNNClassifier(nn.Module):

    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.rnn = nn.RNN(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        embedded = self.embedding(x)  # (batch_size, seq_len, embed_dim)
        _, hidden = self.rnn(embedded)  # hidden: (1, batch_size, hidden_dim)
        output = self.fc(hidden.squeeze(0))  # (batch_size, 1)
        return output.squeeze(1)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = RNNClassifier(
    vocab_size=enc.n_vocab, embed_dim=64, hidden_dim=64
).to(device)

In [12]:
# 8. Train the model using BCEWithLogitsLoss and Adam optimizer
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

EPOCHS = 5
for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0

    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)

        optimizer.zero_grad()
        predictions = model(batch_x)
        loss = criterion(predictions, batch_y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch}/{EPOCHS} | Loss: {avg_loss:.4f}")

Epoch 1/5 | Loss: 0.6985
Epoch 2/5 | Loss: 0.6798
Epoch 3/5 | Loss: 0.6610
Epoch 4/5 | Loss: 0.6507
Epoch 5/5 | Loss: 0.6214


In [16]:
# 9. Evaluate the trained model on the test set and report accuracy
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for batch_x, batch_y in test_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)

        # Forward pass
        logits = model(batch_x)
        # Apply sigmoid to get probabilities, then threshold at 0.5
        predictions = (torch.sigmoid(logits) >= 0.5).float()

        correct += (predictions == batch_y).sum().item()
        total += batch_y.size(0)

accuracy = (correct / total) * 100
print(f"Test Accuracy: {accuracy:.2f}%\n")

Test Accuracy: 60.00%



In [17]:
# 10. Test the model on new review sentences
custom_reviews = [
    "This movie was an absolute masterpiece with brilliant acting!",
    "Terrible plot, awful acting, and a complete waste of time.",
    "A truly heartwarming and beautifully directed story.",
]

print("--- Custom Review Predictions ---")
model.eval()
with torch.no_grad():
    for review in custom_reviews:
        # Tokenize and pad/truncate using the same pipeline
        tokens = enc.encode(review)
        padded_tokens = pad_truncate(tokens)

        # Convert to tensor and add batch dimension: (1, max_len)
        input_tensor = (
            torch.tensor(padded_tokens, dtype=torch.long)
            .unsqueeze(0)
            .to(device)
        )

        logit = model(input_tensor)
        prob = torch.sigmoid(logit).item()
        sentiment = "Positive" if prob >= 0.5 else "Negative"

        print(f"Review: '{review}'")
        print(f"Predicted Sentiment: {sentiment} (Confidence: {prob:.4f})\n")

--- Custom Review Predictions ---
Review: 'This movie was an absolute masterpiece with brilliant acting!'
Predicted Sentiment: Positive (Confidence: 0.9898)

Review: 'Terrible plot, awful acting, and a complete waste of time.'
Predicted Sentiment: Negative (Confidence: 0.0009)

Review: 'A truly heartwarming and beautifully directed story.'
Predicted Sentiment: Positive (Confidence: 0.9712)



In [15]:
# 5. Pre-pad instead of post-pad (critical for standard RNNs)
MAX_LEN = 100
PAD_TOKEN = 0


def pad_truncate(seq, max_len=MAX_LEN):
    if len(seq) > max_len:
        return seq[-max_len:]  # Keep last tokens
    return [PAD_TOKEN] * (max_len - len(seq)) + seq  # Pre-pad with zeros


train_padded = [pad_truncate(seq) for seq in train_tokens]
test_padded = [pad_truncate(seq) for seq in test_tokens]

# 6. Re-create tensors & DataLoaders
X_train = torch.tensor(train_padded, dtype=torch.long)
y_train = torch.tensor(train_labels, dtype=torch.float32)
X_test = torch.tensor(test_padded, dtype=torch.long)
y_test = torch.tensor(test_labels, dtype=torch.float32)

train_loader = DataLoader(
    TensorDataset(X_train, y_train), batch_size=32, shuffle=True
)
test_loader = DataLoader(TensorDataset(X_test, y_test), batch_size=32)

# 7. Re-initialize model
model = RNNClassifier(
    vocab_size=enc.n_vocab, embed_dim=64, hidden_dim=64
).to(device)

# 8. Train with gradient clipping
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.002)

EPOCHS = 12
for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0

    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)

        optimizer.zero_grad()
        predictions = model(batch_x)
        loss = criterion(predictions, batch_y)
        loss.backward()

        # Gradient clipping prevents exploding/vanishing gradient stalls
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch}/{EPOCHS} | Loss: {avg_loss:.4f}")

Epoch 1/12 | Loss: 0.6959
Epoch 2/12 | Loss: 0.6553
Epoch 3/12 | Loss: 0.5868
Epoch 4/12 | Loss: 0.4748
Epoch 5/12 | Loss: 0.3615
Epoch 6/12 | Loss: 0.2346
Epoch 7/12 | Loss: 0.1382
Epoch 8/12 | Loss: 0.0749
Epoch 9/12 | Loss: 0.0432
Epoch 10/12 | Loss: 0.0159
Epoch 11/12 | Loss: 0.0093
Epoch 12/12 | Loss: 0.0072
